
# Topic 1 — SFT + Fine-Tuning Fundamentals

**Core question:** *How do I teach a model a task?*

**Running example (used across all 6 topics in this series):** an Amazon-style
AI shopping / customer-support assistant. A customer describes a problem
("my order hasn't shipped", "I was double-charged", ...) and the assistant
should produce the right support response.

In this notebook we fine-tune a small model — `Qwen3-0.6B`, loaded via
[Unsloth](https://github.com/unslothai/unsloth) — on 500 synthetic
(problem → response) examples using **Supervised Fine-Tuning (SFT)**, and
measure the difference between the base model and the fine-tuned model.

## The mnemonic to remember

> **D-T-F-L-B-O-U**: **D**ataset → **T**okenizer → **F**orward pass → **L**oss
> → **B**ackprop → **O**ptimizer → **U**pdated weights

Every fine-tuning method in this series (LoRA, QLoRA, continued pretraining,
DPO, GRPO, distillation) is a variation on this same loop. Topic 1 is about
getting this loop crystal clear before we start varying it.

```mermaid
flowchart LR
    A[Dataset<br/>500 problem→response pairs] --> B[Tokenizer<br/>text → token IDs]
    B --> C[Forward pass<br/>model predicts next tokens]
    C --> D[Loss<br/>cross-entropy vs target tokens]
    D --> E[Backprop<br/>gradients w.r.t. every weight]
    E --> F[Optimizer step<br/>AdamW updates weights]
    F --> G[Updated weights]
    G -. repeat for every batch, every epoch .-> A
```

### Can you answer this? (keep these in mind as you go)
1. Why does SFT work?
2. What is the loss actually training on?
3. What happens when the dataset is too small?
4. Why can aggressive fine-tuning hurt general capabilities?

We'll answer each one explicitly at the end, backed by what you observe in
this notebook's output.


In [1]:

# --- Formatting helpers (used throughout this notebook) ------------------
# `rich` is a Python library for nicely formatted terminal/notebook output:
# colored text, tables, panels. We use it purely for *readability* of the
# training-material output below -- it has nothing to do with fine-tuning
# itself.
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()

def banner(title: str, subtitle: str = "", style: str = "bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))

def section(title: str):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")

def kv_table(title: str, rows: dict, caption: str = ""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 1: SFT + Fine-Tuning Fundamentals",
       "Amazon-style AI shopping assistant — running example for this series")


╔══════════════════════════════════════════════════════════════════════╗
║ Topic 1: SFT + Fine-Tuning Fundamentals                              ║
║ Amazon-style AI shopping assistant — running example for this series ║
╚══════════════════════════════════════════════════════════════════════╝


## Concept primer (part 1): what "fine-tuning a task" means

**Instruction tuning** means teaching a pretrained model to follow an
instruction/request and produce the desired response — rather than just
continuing text statistically. A base model that has only seen raw internet
text might *complete* "My order hasn't arrived, what should I" with more
rambling customer complaints, because that's what it has seen most often.
After instruction tuning it learns: *this shape of input expects a helpful
response in this shape*.

**Prompt/completion vs. conversational datasets** — two common dataset
shapes:
- *Prompt/completion*: `{"prompt": "...", "completion": "..."}` — one input,
  one expected output. Our dataset is this shape: `{"problem": ..., "response": ...}`.
- *Conversational*: a list of `{"role": ..., "content": ...}` turns (system /
  user / assistant), used for multi-turn chat models. We will convert our
  prompt/completion pairs into this shape below, because that's what modern
  chat models (including Qwen3) expect.

**Chat templates** are a fixed string format a specific model's tokenizer
uses to mark where each turn starts/ends (e.g. `<|im_start|>user ... <|im_end|>`
for Qwen models). Every chat model has its own template baked into its
tokenizer config — if you feed it the wrong format, it still "works" but
produces worse output, because it wasn't trained on that layout. We'll
render our examples through Qwen3's real chat template below.

**Library primer — `datasets` (Hugging Face):** a library for loading and
processing data sets in a standard columnar format (`Dataset`), with fast
indexing/filtering/mapping, independent of the model you'll train. Think of
it as "pandas for ML training data."



## Concept primer (part 2): the training knobs

**Cross-entropy loss** — at each position in the target response, the model
outputs a probability distribution over the entire vocabulary (~150k tokens
for Qwen3) for "what comes next." Cross-entropy loss measures how much
probability mass the model put on the *actual* next token versus everything
else. Low loss = the model was confident in the right token. **The loss is
computed only over the response tokens, not the prompt** — we don't want to
train the model to predict the user's own question.

**Epochs** — one epoch = one full pass over the training dataset. Too few
epochs and the model hasn't learned the pattern yet (underfitting); too many
and it starts memorizing the training examples verbatim instead of
generalizing (overfitting).

**Batch size** — how many examples are processed together before one
weight-update step. Bigger batches give a smoother (less noisy) gradient
estimate but use more memory.

**Learning rate (LR)** — how big a step the optimizer takes in the direction
that reduces loss. Too high → training diverges or oscillates. Too low →
training is painfully slow or gets stuck.

**Gradient accumulation** — simulates a larger batch size on limited memory
by summing gradients over several small "micro-batches" before actually
stepping the optimizer. `effective_batch_size = batch_size * grad_accum_steps`.

**Overfitting** — the model's loss on *training* data keeps dropping, but
its performance on *new, unseen* inputs stops improving or gets worse,
because it has started memorizing specific examples rather than the general
pattern.

**Catastrophic forgetting** — when fine-tuning on a narrow task causes the
model to get *worse* at other things it used to be able to do, because the
weight updates needed for the new task "overwrote" weights that encoded
unrelated general knowledge. This gets worse with: more aggressive learning
rates, more epochs, smaller/narrower datasets, and full fine-tuning (every
weight is editable) vs. parameter-efficient methods (Topic 2).

**Full fine-tuning vs. pretraining** — pretraining teaches the model
language and world knowledge from a massive, generic text corpus using
next-token prediction (self-supervised, no labeled "correct answers").
Fine-tuning (this notebook) starts from an *already pretrained* model and
adjusts it with a much smaller, task-specific *labeled* dataset — we're
steering a model that already knows how to write English towards a specific
behavior, not teaching it English from scratch.

**Library primer — `transformers`:** Hugging Face's library providing
model architectures (loading pretrained weights) and tokenizers in a unified
API across hundreds of model families.

**Library primer — `trl` (Transformer Reinforcement Learning):** Hugging
Face's library of *post-training* trainers built on top of `transformers` —
`SFTTrainer` (this notebook), `DPOTrainer`, `GRPOTrainer` (Topic 4),
`DistillationTrainer` (Topic 5).

**Library primer — `peft` / `bitsandbytes` / `unsloth`:** covered in depth
in Topic 2 (LoRA/QLoRA). In short: `peft` implements parameter-efficient
fine-tuning methods like LoRA; `bitsandbytes` implements low-precision
(4-bit/8-bit) quantization; `unsloth` is a library that patches
`transformers`/`peft` training code to run 2x faster and use less memory on
a single consumer/Colab GPU — it's a performance layer, not a new training
algorithm.


In [2]:

# --- Environment check -----------------------------------------------------
# Honest about what hardware is actually available, since the GPU-dependent
# cells below behave differently depending on this.
import sys
import torch

section("Environment check")

kv_table("Runtime", {
    "Python": sys.version.split()[0],
    "PyTorch": torch.__version__,
    "CUDA available": torch.cuda.is_available(),
    "Apple MPS available": getattr(torch.backends, "mps", None) and torch.backends.mps.is_available(),
})

HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    console.print(f"[bold green]✓ CUDA GPU detected:[/bold green] {gpu_name} ({gpu_mem_gb:.1f} GB)")
else:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected on this machine.[/bold yellow]\n\n"
        "Unsloth and bitsandbytes require a CUDA GPU. The cells below that need\n"
        "real model loading / training / GPU inference will detect this and\n"
        "print a clear [bold]SKIPPED[/bold] message instead of failing -- run this\n"
        "notebook on Google Colab with a T4 GPU (Runtime -> Change runtime type -> GPU)\n"
        "to execute those cells for real.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

            Runtime             
                                
  Key                   Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Python                3.12.5  
  PyTorch               2.14.1  
  CUDA available        False   
  Apple MPS available   True

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected on this machine.                                                                           ┃
┃                                                                                                                 ┃
┃ Unsloth and bitsandbytes require a CUDA GPU. The cells below that need                                          ┃
┃ real model loading / training / GPU inference will detect this and                                              ┃
┃ print a clear SKIPPED message instead of failing -- run this                                                    ┃
┃ notebook on Google Colab with a T4 GPU (Runtime -> Change runtime type -> GPU)                                  ┃
┃ to execute those cells for real.                                                                                ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

# --- Installs ---------------------------------------------------------------
# Safe to re-run: pip skips anything already satisfied. On Colab this installs
# fresh; locally (this environment) these are already present in the project
# virtualenv except unsloth/bitsandbytes, which this cell still attempts so the
# notebook behaves identically on Colab.
import sys

IN_COLAB = "google.colab" in sys.modules

packages = [
    "unsloth",
    "transformers",
    "trl",
    "peft",
    "bitsandbytes",
    "torch",
    "datasets",
    "rich",
]

if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages)}")
    console.print("[green]Installed packages for Colab.[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv "
                  "already has these installed (see requirements.txt). "
                  "Skipping pip install to save time.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these installed (see requirements.txt). 
Skipping pip install to save time.

In [4]:
# --- Load the dataset -------------------------------------------------------
import json
import random
from pathlib import Path
from collections import Counter

section("Loading the dataset")

# Different notebook runners (Colab, plain Jupyter, VS Code's Jupyter extension,
# etc.) can launch the kernel with very different working directories -- so we
# don't just guess one or two relative paths. VS Code's Jupyter extension
# exposes the open notebook's own file path as `__vsc_ipynb_file__`; Colab/plain
# Jupyter don't set that, so we fall back to searching near the current working
# directory instead.
vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]  # also check one level up

RELATIVE_CANDIDATES = [
    "data/support_tickets.json",
    "01_sft_fundamentals/data/support_tickets.json",
]

DATA_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH:
        break

if DATA_PATH is None:
    # Self-sufficient fallback: on Colab (or any machine where only this .ipynb
    # was uploaded, without the data/ folder next to it), regenerate the exact
    # same deterministic dataset in-memory instead of requiring a manual upload.
    console.print("[dim]data/support_tickets.json not found nearby -- generating it in-memory "
                  "instead of requiring a separate file upload.[/dim]")

    PRODUCTS = ["wireless earbuds", "coffee maker", "running shoes", "laptop stand",
                "desk lamp", "yoga mat", "bluetooth speaker", "phone case",
                "winter jacket", "kitchen blender", "backpack", "office chair",
                "water bottle", "gaming mouse", "electric toothbrush", "air fryer",
                "board game", "children's book", "garden hose", "car phone mount"]
    NAMES = ["Alex", "Jordan", "Priya", "Sam", "Taylor", "Morgan", "Chris",
             "Jamie", "Riley", "Casey", "Drew", "Avery"]

    def _order(): return f"#{random.randint(100000, 999999)}"
    def _days(lo=1, hi=14): return random.randint(lo, hi)

    def order_status():
        p, o, d = random.choice(PRODUCTS), _order(), _days(1, 10)
        return (f"I ordered a {p} with order number {o} {d} days ago and I still don't see any "
                f"shipping update. Can you tell me what's going on?",
                f"Thanks for reaching out about order {o}. I checked and your {p} is currently being "
                f"prepared at our fulfillment center and is on track to ship within 1-2 business days. "
                f"You'll receive a tracking email the moment it ships, and estimated delivery is "
                f"{_days(2,5)} days after that. I'll flag the order so it's monitored closely.")

    def returns_refunds():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I want to return the {p} from order {o} because it's not what I expected. How do I "
                f"get a refund?",
                f"No problem at all -- I've started a return for the {p} in order {o}. You'll get a "
                f"prepaid return label by email within the hour. Once the item is scanned by the "
                f"carrier, your refund of the full purchase price will be issued to your original "
                f"payment method within 3-5 business days.")

    def damaged_item():
        p, o = random.choice(PRODUCTS), _order()
        return (f"The {p} I received in order {o} arrived damaged -- it looks like it was crushed in "
                f"shipping. What can you do?",
                f"I'm sorry the {p} arrived damaged, that's not the experience we want for you. I've "
                f"issued a free replacement for order {o} at no extra cost, and you don't need to "
                f"return the damaged item. The replacement will ship within 1 business day with "
                f"expedited shipping.")

    def payment_issue():
        o = _order()
        amount = random.choice([19.99, 29.99, 49.50, 89.00, 124.75])
        return (f"I was charged ${amount} twice for order {o}. Can you check my payment and fix this?",
                f"I checked order {o} and confirmed a duplicate charge of ${amount}. I've submitted a "
                f"refund for the extra charge, which will appear on your statement within 3-5 business "
                f"days. Only one charge of ${amount} will remain for your order.")

    def account_access():
        name = random.choice(NAMES)
        return (f"Hi, this is {name}. I can't log into my account, it keeps saying my password is "
                f"incorrect even after I reset it. Can you help?",
                f"Hi {name}, sorry for the trouble logging in. I've sent a fresh password reset link "
                f"to your account email -- it will be valid for 30 minutes. If it still doesn't work "
                f"after that, it may be a browser cache issue, so try an incognito window or clearing "
                f"cookies for our site.")

    def shipping_delay():
        p, o, d = random.choice(PRODUCTS), _order(), _days(5, 14)
        return (f"My {p} from order {o} was supposed to arrive {d} days ago and it still hasn't shown "
                f"up. This is really frustrating.",
                f"I completely understand the frustration, and I'm sorry for the delay on your {p} in "
                f"order {o}. I checked with the carrier and the package is delayed in transit but "
                f"still moving -- updated delivery is within 2 days. As an apology for the delay, I've "
                f"credited your account with a $10 voucher for your next order.")

    def cancellation():
        p, o = random.choice(PRODUCTS), _order()
        return (f"I need to cancel my order {o} for the {p} -- I accidentally ordered two of them. Is "
                f"it too late?",
                f"Good news -- order {o} hasn't shipped yet, so I was able to cancel it for you right "
                f"away. You will not be charged for this order, and any pending authorization will "
                f"drop off within a few business days. The other order for the {p} is unaffected.")

    def product_question():
        p = random.choice(PRODUCTS)
        return (f"Before I buy the {p}, can you tell me if it comes with a warranty and what the "
                f"return window is?",
                f"Great question! The {p} comes with a 1-year manufacturer warranty covering defects, "
                f"and you can return it within 30 days of delivery for any reason as long as it's in "
                f"like-new condition. If anything goes wrong after the return window, our support "
                f"team can still help with the manufacturer warranty.")

    def subscription_issue():
        return ("I signed up for a monthly subscription but I was charged again this month even "
                "though I thought I canceled it last week. Can you check?",
                "I looked into your subscription and I can confirm the cancellation request from "
                "last week did not fully process due to a billing sync issue on our end. I've now "
                "canceled the subscription effective immediately and refunded this month's charge "
                "in full -- it will reflect within 3-5 business days.")

    def promo_discount():
        code = random.choice(["SAVE10", "WELCOME15", "SUMMER20", "FIRST5"])
        return (f"I tried to use the promo code {code} at checkout but it says invalid. The order "
                f"total didn't get discounted. Can you help?",
                f"Thanks for flagging this. I checked the promo code {code} and applied the discount "
                f"manually to your most recent order, so you'll see the adjusted total reflected "
                f"shortly. In the future, that code works best when applied before adding a gift wrap "
                f"option, which was conflicting with it at checkout.")

    CATEGORIES = {
        "order_status": order_status, "returns_refunds": returns_refunds,
        "damaged_item": damaged_item, "payment_issue": payment_issue,
        "account_access": account_access, "shipping_delay": shipping_delay,
        "cancellation": cancellation, "product_question": product_question,
        "subscription_issue": subscription_issue, "promo_discount": promo_discount,
    }

    def generate_support_tickets(n=500, seed=42):
        rng_state = random.getstate()
        random.seed(seed)
        names = list(CATEGORIES.keys())
        per, rem = n // len(names), n % len(names)
        examples = []
        for i, name in enumerate(names):
            count = per + (1 if i < rem else 0)
            for _ in range(count):
                problem, response = CATEGORIES[name]()
                examples.append({"category": name, "problem": problem, "response": response})
        random.shuffle(examples)
        random.setstate(rng_state)
        return examples

    raw_examples = generate_support_tickets()
    DATA_PATH = Path.cwd() / "data" / "support_tickets.json"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_PATH.open("w") as f:
        json.dump(raw_examples, f, indent=2)
else:
    with DATA_PATH.open() as f:
        raw_examples = json.load(f)

console.print(f"Loaded [bold]{len(raw_examples)}[/bold] examples from {DATA_PATH}")

cat_counts = Counter(ex["category"] for ex in raw_examples)
table = Table(title="Category distribution", box=box.SIMPLE_HEAVY)
table.add_column("Category")
table.add_column("Count", justify="right")
for cat, n in sorted(cat_counts.items()):
    table.add_row(cat, str(n))
console.print(table)

console.print("\n[bold]Sample example:[/bold]")
sample = raw_examples[0]
console.print(Panel(f"[bold]Category:[/bold] {sample['category']}\n\n"
                     f"[bold]Problem (user):[/bold]\n{sample['problem']}\n\n"
                     f"[bold]Response (assistant):[/bold]\n{sample['response']}",
                     box=box.ROUNDED))


── Loading the dataset ─────────────────────────────────────────

Loaded 500 examples from 
/Users/bhanupraka.gopularam/aiprojects/llm-fine-tuning/01_sft_fundamentals/data/support_tickets.json

    Category distribution     
                              
  Category             Count  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  account_access          50  
  cancellation            50  
  damaged_item            50  
  order_status            50  
  payment_issue           50  
  product_question        50  
  promo_discount          50  
  returns_refunds         50  
  shipping_delay          50  
  subscription_issue      50

Sample example:

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Category: subscription_issue                                                                                    │
│                                                                                                                 │
│ Problem (user):                                                                                                 │
│ I signed up for a monthly subscription but I was charged again this month even though I thought I canceled it   │
│ last week. Can you check?                                                                                       │
│                                                                                                                 │
│ Response (assistant):                                                                                           │
│ I looked into your subscription and I can confirm the cancellation request from last week did not fully process │
│ due to a billing sync issue on our end. I've now canceled the subscription effective immediately and refunded   │
│ this month's charge in full — it will reflect within 3-5 business days.                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## From (problem, response) pairs to a chat-formatted training example

Our raw data is prompt/completion shaped: `{"problem": ..., "response": ...}`.
To train a chat model we convert each pair into a **conversation** with a
system message (sets the assistant's role), a user turn (the customer's
problem) and an assistant turn (the target response) — then run it through
the model's own chat template so the tokens line up exactly the way the
model was pretrained to expect them.


In [5]:

# --- Train/held-out split + chat template -----------------------------------
import random

random.seed(7)
shuffled = raw_examples[:]
random.shuffle(shuffled)

N_HOLDOUT = 10
holdout_examples = shuffled[:N_HOLDOUT]          # used for before/after comparison
train_examples = shuffled[N_HOLDOUT:]             # used for SFT training

console.print(f"Training examples: [bold]{len(train_examples)}[/bold]  |  "
              f"Held-out (never trained on) examples: [bold]{len(holdout_examples)}[/bold]")

SYSTEM_PROMPT = (
    "You are a helpful customer support assistant for an online shopping "
    "platform. Respond to the customer's problem clearly and helpfully."
)

def to_conversation(example):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["response"]},
    ]

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"   # tokenizer is small to download even without a GPU
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

demo_conv = to_conversation(train_examples[0])
templated = tokenizer.apply_chat_template(demo_conv, tokenize=False)

section("Chat template demo")
console.print("[bold]Before templating (raw conversation list):[/bold]")
console.print(demo_conv)
console.print("\n[bold]After templating (what the model actually sees as text):[/bold]")
console.print(Panel(templated, box=box.ROUNDED))

n_tokens = len(tokenizer(templated)["input_ids"])
console.print(f"\nThis example is [bold]{n_tokens}[/bold] tokens long once templated.")


Training examples: 490  |  Held-out (never trained on) examples: 10

── Chat template demo ──────────────────────────────────────────

Before templating (raw conversation list):

[
    {
        'role': 'system',
        'content': "You are a helpful customer support assistant for an online shopping platform. Respond to the 
customer's problem clearly and helpfully."
    },
    {
        'role': 'user',
        'content': "My running shoes from order #245095 was supposed to arrive 7 days ago and it still hasn't shown
up. This is really frustrating."
    },
    {
        'role': 'assistant',
        'content': "I completely understand the frustration, and I'm sorry for the delay on your running shoes in 
order #245095. I checked with the carrier and the package is delayed in transit but still moving — updated delivery
is within 2 days. As an apology for the delay, I've credited your account with a $10 voucher for your next order."
    }
]

After templating (what the model actually sees as text):

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful customer support assistant for an online shopping platform. Respond to the customer's problem │
│ clearly and helpfully.<|im_end|>                                                                                │
│ <|im_start|>user                                                                                                │
│ My running shoes from order #245095 was supposed to arrive 7 days ago and it still hasn't shown up. This is     │
│ really frustrating.<|im_end|>                                                                                   │
│ <|im_start|>assistant                                                                                           │
│ <think>                                                                                                         │
│                                                                                                                 │
│ </think>                                                                                                        │
│                                                                                                                 │
│ I completely understand the frustration, and I'm sorry for the delay on your running shoes in order #245095. I  │
│ checked with the carrier and the package is delayed in transit but still moving — updated delivery is within 2  │
│ days. As an apology for the delay, I've credited your account with a $10 voucher for your next order.<|im_end|> │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

This example is 150 tokens long once templated.

In [6]:

# --- Load the model via Unsloth (GPU-only) ----------------------------------
# Unsloth patches transformers/peft loading & training to run faster and use
# less memory. FastLanguageModel.from_pretrained() returns a (model, tokenizer)
# pair ready for training.
MAX_SEQ_LENGTH = 512

if HAS_CUDA:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,   # full-precision for this fundamentals demo; see Topic 2 for 4-bit (QLoRA)
    )
    console.print("[bold green]✓ Model loaded via Unsloth.[/bold green]")
else:
    model = None
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- loading Qwen3-0.6B via Unsloth requires a CUDA GPU.\n"
        "Expected on Colab: a few seconds to download + load the model, then a\n"
        "confirmation that the model and tokenizer are ready for training.",
        title="Model load (GPU-only)", style="yellow",
    ))


╭───────────────────────────────────────────── Model load (GPU-only) ─────────────────────────────────────────────╮
│ SKIPPED -- loading Qwen3-0.6B via Unsloth requires a CUDA GPU.                                                  │
│ Expected on Colab: a few seconds to download + load the model, then a                                           │
│ confirmation that the model and tokenizer are ready for training.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:

# --- Baseline inference: how does the UN-tuned model answer? ----------------
def generate_response(model, tokenizer, problem, max_new_tokens=120):
    conv = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    prompt_text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()

baseline_outputs = {}

if HAS_CUDA:
    FastLanguageModel.for_inference(model)
    section("Baseline (before fine-tuning) responses")
    for ex in holdout_examples[:3]:
        out = generate_response(model, tokenizer, ex["problem"])
        baseline_outputs[ex["problem"]] = out
        console.print(Panel(f"[bold]Problem:[/bold] {ex['problem']}\n\n"
                             f"[bold]Base model response:[/bold]\n{out}", box=box.ROUNDED))
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- generating from the base model requires the model loaded above.\n"
        "Expected on Colab: the un-tuned model's replies to 3 held-out customer problems,\n"
        "typically generic, sometimes off-topic or missing the expected tone/structure.",
        title="Baseline inference (GPU-only)", style="yellow",
    ))


╭───────────────────────────────────────── Baseline inference (GPU-only) ─────────────────────────────────────────╮
│ SKIPPED -- generating from the base model requires the model loaded above.                                      │
│ Expected on Colab: the un-tuned model's replies to 3 held-out customer problems,                                │
│ typically generic, sometimes off-topic or missing the expected tone/structure.                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Configuring `SFTTrainer`

| Hyperparameter | What it controls | If you increase it... | If you decrease it... |
|---|---|---|---|
| `num_train_epochs` | passes over the data | risk of overfitting/memorization goes up | may underfit, pattern not learned |
| `per_device_train_batch_size` | examples per step | smoother gradients, more memory | noisier gradients, less memory |
| `gradient_accumulation_steps` | micro-batches summed before a step | larger effective batch size without more memory | smaller effective batch, more frequent (noisier) updates |
| `learning_rate` | optimizer step size | faster learning, risk of divergence/instability | slower learning, risk of getting stuck |
| `max_seq_length` | longest sequence trained on | handles longer conversations, more memory/time per step | truncates long examples, less memory |

For this small 490-example, short-response dataset we deliberately pick a
**short run** (few epochs, modest LR) — the goal is to clearly observe the
before/after difference, not to squeeze out maximum quality.


In [8]:

# --- SFTTrainer configuration ------------------------------------------------
# Building the config objects is plain Python/dataclass construction -- it
# works with or without a GPU. Only the actual trainer.train() call below
# needs CUDA.
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="outputs/sft_fundamentals",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_length=MAX_SEQ_LENGTH,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    # transformers validates bf16/fp16 against available hardware AT
    # CONSTRUCTION TIME (not just at trainer.train()), so this config must
    # explicitly know whether a GPU is present even though we're only
    # building it here, not training yet.
    use_cpu=not HAS_CUDA,
    bf16=HAS_CUDA,
)

kv_table("SFTConfig (this run)", {
    "epochs": sft_config.num_train_epochs,
    "per_device_train_batch_size": sft_config.per_device_train_batch_size,
    "gradient_accumulation_steps": sft_config.gradient_accumulation_steps,
    "effective_batch_size": sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps,
    "learning_rate": sft_config.learning_rate,
    "max_length": sft_config.max_length,
})


          SFTConfig (this run)          
                                        
  Key                           Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  epochs                        3       
  per_device_train_batch_size   4       
  gradient_accumulation_steps   2       
  effective_batch_size          8       
  learning_rate                 0.0002  
  max_length                    512

In [9]:

# --- Build the training dataset + train -------------------------------------
from datasets import Dataset

def to_text_example(example):
    return {"text": tokenizer.apply_chat_template(to_conversation(example), tokenize=False)}

train_dataset = Dataset.from_list([to_text_example(ex) for ex in train_examples])
console.print(f"Training dataset ready: {train_dataset}")

if HAS_CUDA:
    from trl import SFTTrainer

    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        args=sft_config,
    )
    section("Training")
    train_result = trainer.train()
    kv_table("Training result", {
        "final loss": train_result.training_loss,
        "steps": train_result.global_step,
    })
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- actual training requires a CUDA GPU.\n"
        "Expected on Colab (T4): a progress bar over ~3 epochs x ~123 steps,\n"
        "training loss printed every 5 steps, generally dropping from roughly\n"
        "2.5-3.0 down toward 0.3-0.8 by the end of training for a dataset this size.",
        title="SFT training (GPU-only)", style="yellow",
    ))


Training dataset ready: Dataset({
    features: ['text'],
    num_rows: 490
})

╭──────────────────────────────────────────── SFT training (GPU-only) ────────────────────────────────────────────╮
│ SKIPPED -- actual training requires a CUDA GPU.                                                                 │
│ Expected on Colab (T4): a progress bar over ~3 epochs x ~123 steps,                                             │
│ training loss printed every 5 steps, generally dropping from roughly                                            │
│ 2.5-3.0 down toward 0.3-0.8 by the end of training for a dataset this size.                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:

# --- Post-training inference + before/after comparison ----------------------
if HAS_CUDA:
    FastLanguageModel.for_inference(model)
    section("Fine-tuned (after SFT) responses -- same held-out problems")
    for ex in holdout_examples[:3]:
        before = baseline_outputs.get(ex["problem"], "(not captured)")
        after = generate_response(model, tokenizer, ex["problem"])
        table = Table(title=f"Problem: {ex['problem'][:70]}...", box=box.SIMPLE_HEAVY, show_lines=True)
        table.add_column("Before SFT (base model)", style="red", width=50)
        table.add_column("After SFT (fine-tuned)", style="green", width=50)
        table.add_column("Expected (dataset)", style="cyan", width=50)
        table.add_row(before, after, ex["response"])
        console.print(table)
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- post-training inference requires the trained model above.\n"
        "Expected on Colab: the fine-tuned model's replies should closely match the\n"
        "style/structure/tone of the dataset's target responses (empathetic opener,\n"
        "concrete resolution, specific timeframe) far more consistently than the base\n"
        "model's generic replies captured earlier.",
        title="Post-training comparison (GPU-only)", style="yellow",
    ))


╭────────────────────────────────────── Post-training comparison (GPU-only) ──────────────────────────────────────╮
│ SKIPPED -- post-training inference requires the trained model above.                                            │
│ Expected on Colab: the fine-tuned model's replies should closely match the                                      │
│ style/structure/tone of the dataset's target responses (empathetic opener,                                      │
│ concrete resolution, specific timeframe) far more consistently than the base                                    │
│ model's generic replies captured earlier.                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## This at Big-Tech scale: ~100 users vs. ~10k users

Fine-tuning itself is a one-time (or periodic) training job, but what
changes with scale is **how often you retrain, how you validate before
shipping, and the inference cost once the fine-tuned model is serving
traffic.**


In [11]:

section("Scale discussion: ~100 users vs ~10k users")

table = Table(title="SFT fine-tuning at scale (Qwen3-0.6B-class model)", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Concern")
table.add_column("~100 users", style="cyan")
table.add_column("~10k users", style="magenta")

rows = [
    ("Training cost", "Single T4/A10 GPU, < $1-5 per retrain, minutes to run",
                      "Still a small model -- a few A100-hours per retrain; the cost center shifts to data curation and eval, not raw compute"),
    ("Retrain cadence", "Ad hoc, whenever support playbooks change",
                        "Scheduled retrains (e.g. weekly) as new support categories/edge cases accumulate"),
    ("Data volume needed", "500 examples is plausible and sufficient to start",
                           "Needs thousands-to-tens-of-thousands of real (sampled + reviewed) support interactions to cover the long tail of categories"),
    ("Serving latency", "A single instance easily absorbs bursty traffic; no autoscaling needed",
                        "Needs autoscaled inference replicas, request batching, and p95/p99 latency SLOs (Topic 6)"),
    ("Memory / infra", "One GPU, maybe even CPU-only for a 0.6B model at low QPS",
                       "Multiple GPU replicas behind a load balancer; GPU utilization becomes a cost lever"),
    ("Operational concerns", "Manual before/after spot-checks (like this notebook) are enough",
                            "Needs automated regression eval suites, canary rollout, and rollback plans before every retrain reaches 100% of traffic (Topic 6)"),
]
for concern, small, big in rows:
    table.add_row(concern, small, big)

console.print(table)


── Scale discussion: ~100 users vs ~10k users ──────────────────

                                 SFT fine-tuning at scale (Qwen3-0.6B-class model)                                 
                                                                                                                   
  Concern                ~100 users                                   ~10k users                                   
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Training cost          Single T4/A10 GPU, < $1-5 per retrain,       Still a small model -- a few A100-hours per  
                         minutes to run                               retrain; the cost center shifts to data      
                                                                      curation and eval, not raw compute           
                                                                                                                   
  Retrain cadence        Ad hoc, whenever support playbooks change    Scheduled retrains (e.g. weekly) as new      
                                                                      support categories/edge cases accumulate     
                                                                                                                   
  Data volume needed     500 examples is plausible and sufficient     Needs thousands-to-tens-of-thousands of      
                         to start                                     real (sampled + reviewed) support            
                                                                      interactions to cover the long tail of       
                                                                      categories                                   
                                                                                                                   
  Serving latency        A single instance easily absorbs bursty      Needs autoscaled inference replicas,         
                         traffic; no autoscaling needed               request batching, and p95/p99 latency SLOs   
                                                                      (Topic 6)                                    
                                                                                                                   
  Memory / infra         One GPU, maybe even CPU-only for a 0.6B      Multiple GPU replicas behind a load          
                         model at low QPS                             balancer; GPU utilization becomes a cost     
                                                                      lever                                        
                                                                                                                   
  Operational concerns   Manual before/after spot-checks (like this   Needs automated regression eval suites,      
                         notebook) are enough                         canary rollout, and rollback plans before    
                                                                      every retrain reaches 100% of traffic        
                                                                      (Topic 6)                                   


## Can you answer this? (Topic 1 training checkpoints)

Try to answer each one yourself first, then check the answer.

**1. Why does SFT work?**
<details><summary>Answer</summary>
The base model already encodes language and broad world knowledge from
pretraining. SFT doesn't teach it to write — it narrows the enormous space
of things a language model *could* say down to the specific style, tone and
content pattern demonstrated by your labeled examples, by nudging weights in
the direction that makes the model's own predicted tokens match your target
responses more closely.
</details>

**2. What is the loss actually training on?**
<details><summary>Answer</summary>
Cross-entropy loss over the *response* tokens only (the prompt/user turn is
masked out of the loss). At every response-token position, the model is
penalized for how much probability it assigned to tokens other than the
actual next token in the target response.
</details>

**3. What happens when the dataset is too small?**
<details><summary>Answer</summary>
With too few examples (or too many epochs over a small set), the model can
memorize the *exact* training examples rather than learning the general
pattern behind them — it overfits. You'd see training loss keep dropping
while held-out-example quality plateaus or gets repetitive/overly specific to
training phrasing.
</details>

**4. Why can aggressive fine-tuning hurt general capabilities?**
<details><summary>Answer</summary>
Every weight update nudges *all* the weights involved in producing the
training examples' tokens — there's no built-in mechanism protecting
unrelated capabilities. A high learning rate, many epochs, or a narrow
dataset can overwrite general-purpose weight patterns the model needed for
other tasks (writing code, general reasoning, etc.), which is catastrophic
forgetting. Parameter-efficient methods (Topic 2: LoRA/QLoRA) are one of the
main tools for controlling this blast radius.
</details>


In [12]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Generated/loaded 500 synthetic customer-support examples\n"
    "  2. Converted them to chat-formatted conversations using Qwen3's real chat template\n"
    "  3. (On GPU) loaded Qwen3-0.6B via Unsloth and captured BASELINE responses\n"
    "  4. Configured and ran SFTTrainer with explained hyperparameters\n"
    "  5. (On GPU) compared BEFORE vs AFTER responses on held-out problems\n"
    "  6. Discussed how this scales from ~100 to ~10k users\n\n"
    "[bold]Next: Topic 2 -- LoRA / QLoRA / PEFT[/bold]\n"
    "We reuse this exact dataset and run three experiments on it --\n"
    "Full fine-tuning vs. LoRA vs. QLoRA -- and measure GPU memory, trainable\n"
    "parameters, training time, validation score and adapter size side by side.",
    title="Topic 1 complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔═══════════════════════════════════════════════ Topic 1 complete ════════════════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Generated/loaded 500 synthetic customer-support examples                                                   ║
║   2. Converted them to chat-formatted conversations using Qwen3's real chat template                            ║
║   3. (On GPU) loaded Qwen3-0.6B via Unsloth and captured BASELINE responses                                     ║
║   4. Configured and ran SFTTrainer with explained hyperparameters                                               ║
║   5. (On GPU) compared BEFORE vs AFTER responses on held-out problems                                           ║
║   6. Discussed how this scales from ~100 to ~10k users                                                          ║
║                                                                                                                 ║
║ Next: Topic 2 -- LoRA / QLoRA / PEFT                                                                            ║
║ We reuse this exact dataset and run three experiments on it --                                                  ║
║ Full fine-tuning vs. LoRA vs. QLoRA -- and measure GPU memory, trainable                                        ║
║ parameters, training time, validation score and adapter size side by side.                                      ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝